In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from torchvision import datasets, models
from torchvision.transforms import transforms
from torchvision.utils import save_image

from captum.attr import IntegratedGradients, Occlusion
from captum.attr import visualization as viz

In [8]:
model_path = Path("../models/resnet18_mnist_trained.pth")
data_path = Path("../data/")
raw_data_path = data_path / "MNIST" / "raw"
noised_data_path = data_path / "MNIST" / "noised"

In [7]:
def get_device():
    return torch.device("cuda")


def load_model(model_path, num_classes=10):
    device = get_device()

    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    state_dict = torch.load(Path(model_path), map_location=device)
    model.load_state_dict(state_dict)

    model = model.to(device)
    model.eval()

    return model


def get_test_data(root):
    transform = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ]
    )

    return datasets.MNIST(
        root="../data", download=False, train=False, transform=transform
    )


def get_test_data_loader():
    return DataLoader(
        get_test_data(data_path),
        batch_size=64,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )


def add_gaussian_noise(tensor, sigma=0.1):
    # Generujemy szum o tym samym kształcie co obrazek
    noise = torch.randn_like(tensor) * sigma
    noisy_tensor = tensor + noise
    # Przycinamy wartości do zakresu [0, 1]
    return torch.clamp(noisy_tensor, -1.0, 1.0)


In [10]:
device = get_device()
model = load_model(model_path=model_path)
test_data_loader = get_test_data_loader()